# Conditional Distributions & Cross-Field Constraints

Distributions describe one field at a time; correlations tie fields together linearly. But some relationships are **conditional** — a field's whole *distribution* depends on another field's value — or **ordinal** — several fields must always come out in a fixed order. Neither is expressible with copulas.

Gendantic covers both, with no LLM involved:

- **`Conditional(on=, cases=, default=)`** switches a field's distribution based on another field's value (an exact category, or a numeric `Range` bin).
- **`Constraints(Ordering(...))`** guarantees two or more fields come out sorted ascending per record.

Everything below is deterministic from a seed, so it's reproducible in tests and CI.

In [ ]:
from typing import Annotated

import numpy as np
import matplotlib.pyplot as plt
from pydantic import BaseModel

from gendantic import (
    DistributionSampler,
    LLMDrivenModelAnalyser,
    Conditional,
    Constraints,
    Ordering,
    Range,
    Normal,
    Uniform,
    Categorical,
    fidelity_report,
)

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)

## Generating data without an LLM

In a real pipeline you'd pass a model to `generate_synthetic_data(...)`. Here we sample the distribution fields directly with `DistributionSampler` so the notebook is **self-contained and LLM-free** — conditionals and constraints are applied during sampling either way.

In [ ]:
def sample_records(model, count, seed=42):
    """Sample spec-compliant records without an LLM."""
    specs = LLMDrivenModelAnalyser.extract_distribution_specs(model)
    correlations = getattr(model, '__correlations__', None)
    constraints = getattr(model, '__constraints__', None)
    return DistributionSampler(seed=seed).sample_fields(
        specs, count, correlations=correlations, constraints=constraints
    )

## 1. Conditional on a category

An employee's salary distribution depends on their department. `Eng` and `Sales` each get their own `Normal`; every other department (here `HR`) falls through to `default`.

The discriminator named in `on=` must itself be a distribution-sampled field.

In [ ]:
class Employee(BaseModel):
    department: Annotated[
        str, Categorical(weights={'Eng': 0.5, 'Sales': 0.3, 'HR': 0.2})
    ]
    salary: Annotated[
        float,
        Conditional(
            on='department',
            cases={
                'Eng': Normal(mean=90000, std=15000),
                'Sales': Normal(mean=70000, std=20000),
            },
            default=Normal(mean=50000, std=10000),
        ),
    ]


records = sample_records(Employee, count=4000)
records[0]

The per-department salary means land on each case's target — and the salary histogram is **multi-modal**, one mode per branch, which a single unconditional distribution could never produce.

In [ ]:
by_dept = {}
for r in records:
    by_dept.setdefault(r['department'], []).append(r['salary'])

for dept in ('Eng', 'Sales', 'HR'):
    print(f"{dept:6s} mean salary = {np.mean(by_dept[dept]):>10,.0f}")

fig, ax = plt.subplots()
for dept in ('Eng', 'Sales', 'HR'):
    ax.hist(by_dept[dept], bins=40, alpha=0.6, label=dept)
ax.set_title('salary is conditional on department')
ax.set_xlabel('salary')
ax.set_ylabel('count')
ax.legend()
plt.show()

## 2. Conditional on a numeric threshold

Case keys can be `Range(min, max)` bins instead of exact values. Ranges are **half-open** `[min, max)`, and either bound may be omitted for an open-ended bin. A bonus that grows with age band:

Numeric bins are matched on the **converted** value the record exposes (after int rounding / clipping), so a boundary like `30` behaves exactly as written.

In [ ]:
class Comp(BaseModel):
    age: Annotated[int, Uniform(min=20, max=65)]
    bonus: Annotated[
        float,
        Conditional(
            on='age',
            cases={
                Range(max=30): Normal(mean=2000, std=300),      # age < 30
                Range(30, 50): Normal(mean=5000, std=400),      # 30 <= age < 50
                Range(min=50): Normal(mean=9000, std=500),      # age >= 50
            },
            default=Normal(mean=0, std=1),
        ),
    ]


comp = sample_records(Comp, count=4000, seed=1)

ages = np.array([r['age'] for r in comp])
bonuses = np.array([r['bonus'] for r in comp])

fig, ax = plt.subplots()
ax.scatter(ages, bonuses, s=6, alpha=0.3)
for edge in (30, 50):
    ax.axvline(edge, color='k', ls='--', lw=1)
ax.set_title('bonus steps up at the age-band boundaries')
ax.set_xlabel('age')
ax.set_ylabel('bonus')
plt.show()

## 3. Dependency resolution

A conditional field may depend on **another conditional field**. Gendantic resolves them in dependency order (and raises on cycles or self-references). Here `fee` depends on `rate`, which is itself conditional on `tier`.

In [ ]:
class Account(BaseModel):
    tier: Annotated[str, Categorical(weights={'gold': 0.5, 'silver': 0.5})]
    rate: Annotated[
        float,
        Conditional(
            on='tier',
            cases={'gold': Normal(10, 0.5), 'silver': Normal(20, 0.5)},
            default=Normal(0, 1),
        ),
    ]
    fee: Annotated[
        float,
        Conditional(
            on='rate',
            cases={Range(max=15): Normal(100, 2), Range(min=15): Normal(200, 2)},
            default=Normal(0, 1),
        ),
    ]


acct = sample_records(Account, count=2000, seed=2)
for tier in ('gold', 'silver'):
    fees = [r['fee'] for r in acct if r['tier'] == tier]
    print(f"{tier:6s} -> mean fee = {np.mean(fees):.0f}")

## 4. Cross-field ordering constraints

Declare an `Ordering` inside `__constraints__` to guarantee fields come out sorted ascending per record. A project's `kickoff <= review <= deadline`, always:

In [ ]:
class Project(BaseModel):
    kickoff_day: Annotated[float, Uniform(min=0, max=365)]
    review_day: Annotated[float, Uniform(min=0, max=365)]
    deadline_day: Annotated[float, Uniform(min=0, max=365)]

    __constraints__ = Constraints(
        Ordering('kickoff_day', 'review_day', 'deadline_day'),
    )


projects = sample_records(Project, count=3000, seed=3)
violations = sum(
    1 for p in projects
    if not (p['kickoff_day'] <= p['review_day'] <= p['deadline_day'])
)
print(f'ordering violations: {violations} / {len(projects)}')

### The trade-off: marginals become order statistics

Ordering is enforced by **sorting each row's values across the constrained fields**. That reassigns which field gets which value, so the *combined* pool is preserved but each field's individual marginal shifts: the first field becomes the row-wise minimum, the last the maximum. All three fields below are sampled from the **same** `Uniform(0, 365)`, yet their distributions pull apart.

Use ordering when the invariant matters more than the exact per-field marginals.

In [ ]:
fig, ax = plt.subplots()
for name in ('kickoff_day', 'review_day', 'deadline_day'):
    values = [p[name] for p in projects]
    ax.hist(values, bins=40, alpha=0.5, label=f"{name} (mean {np.mean(values):.0f})")
ax.set_title('same Uniform(0, 365), reshaped into min / median / max by Ordering')
ax.set_xlabel('day of year')
ax.set_ylabel('count')
ax.legend()
plt.show()

### Preserving marginals with `method="resample"`

Sorting is not the only option. When your fields have **different, already-mostly-separated marginals** and you want each to keep its own distribution, use `method="resample"`. It keeps each field's sampled value and redraws only the records that violate the order, repeating until the batch complies.

A hard order and *identical* marginals are mathematically incompatible (if `a <= b` always and both share a distribution, then `a = b`), so resample preserves marginals only when they're compatible with the order. Here `birth < hire < termination` are drawn from disjoint ranges, so every field keeps its own `Uniform`:

In [ ]:
class Career(BaseModel):
    birth: Annotated[float, Uniform(min=0, max=30)]
    hire: Annotated[float, Uniform(min=30, max=60)]
    termination: Annotated[float, Uniform(min=60, max=100)]

    __constraints__ = Constraints(
        Ordering('birth', 'hire', 'termination', method='resample'),
    )


careers = sample_records(Career, count=4000, seed=4)
violations = sum(
    1 for c in careers
    if not (c['birth'] <= c['hire'] <= c['termination'])
)
print(f'ordering violations: {violations} / {len(careers)}')
for name in ('birth', 'hire', 'termination'):
    print(f"{name:12s} mean = {np.mean([c[name] for c in careers]):5.1f}")

Each field's mean sits at its own distribution's midpoint (15 / 45 / 80) — the marginals are **preserved**, not reshaped. Contrast with `sort`, which would pull the three toward the pooled min / median / max. Because the fields are validated per marginal, `fidelity_report` passes:

In [ ]:
career_report = fidelity_report(careers, Career, alpha=0.01)
print(career_report)

If the marginals overlap so heavily that few draws satisfy the order, the rejection budget is exhausted and generation **raises** rather than return silently distorted data — a signal to separate the marginals or fall back to `method='sort'`.

## 5. Validating conditional data

`fidelity_report` checks conditional fields **per case branch**: it groups records by which case matched (on the stored discriminator value) and tests each group against its own case spec, labelling the result `field | group`.

We use `alpha=0.01` for the aggregate verdict here: with many goodness-of-fit checks, the default `0.05` produces the occasional false positive on genuinely correct data.

In [ ]:
report = fidelity_report(records, Employee, alpha=0.01)
print(report)

Each branch's `expected_mean` reflects its own case spec — proof the records were routed to the right distribution.

In [ ]:
for f in report.fields:
    if f.group:
        print(f"{f.group:24s} expected_mean={f.expected_mean:>10,.0f}")

## Recap

- **`Conditional`** switches a field's distribution on another field's value — by exact category or numeric `Range` bin — and resolves chains of dependencies.
- **`Constraints(Ordering(...))`** enforces a hard ascending order across fields, at the cost of reshaping their individual marginals into order statistics.
- **`fidelity_report`** validates conditional fields per branch, so you can assert the routing in tests.

All of it is LLM-free and deterministic from a seed.